In [ ]:
from pathlib import Path
import sys
# Add project root to sys.path
project_root = Path(__file__).resolve().parent.parent if "__file__" in locals() else Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import marimo as mo
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F

from configs.stage3 import Stage3Config
from dataset.latent import LatentPairs
from mini_wa_jepa.visualization.rerun_logger import colorize_bev_4ch
from models.latent import LatentModel
from training.vicreg import latent_loss

# 🧠 阶段 3 核心逻辑一步步拆解（Marimo 交互走读手册）

在 VSCode 里看工程代码，因为要分拆模块、写各种类和配置，很容易让人眼花缭乱。
**在这里，我们将阶段 3 的所有黑盒逻辑像剥洋葱一样，从上往下一行一行拆开给你看：**

1. **数据切片**：真实抽取一对相邻帧 $(t, t+1)$ 与自车速度；
2. **编码器剖析**：看图像怎么从 $128 \times 128$ 压缩成 64 个高阶 Token；
3. **预测器推演**：输入自车动作，模型在脑海里“预测未来”；
4. **VICReg 损失手把手算**：亲眼看方差与协方差公式里的每一个真实数字；
5. **潜空间可视化对比**：用 PCA 把 256 维抽象特征转成肉眼能看懂的彩色特征图！

In [ ]:
# 载入阶段 3 训练好的最佳模型权重
config = Stage3Config()
model = LatentModel(config)
ckpt_path = Path("artifacts/restart_stage3/vicreg_001/best.pt")

if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])
    status_msg = f"✅ 已成功载入阶段 3 训练的最佳 Checkpoint (`best.pt`，Epoch {ckpt.get('epoch', 10)})"
else:
    status_msg = "⚠️ 未找到预训练权重，使用随机初始化权重"

model.eval()

mo.vstack([
    mo.md(f"### 📦 第一步：初始化阶段 3 模型架构 ({status_msg})"),
    mo.md(f"- **编码器参数量**: {sum(p.numel() for p in model.encoder.parameters()):,} (约 6.2M)"),
    mo.md(f"- **预测器参数量**: {sum(p.numel() for p in model.predictor.parameters()):,} (约 11.0M)"),
    mo.md(f"- **总参数量**: {sum(p.numel() for p in model.parameters()):,} (严格符合 15–30M 预算)"),
])

In [ ]:
# 载入真实验证集数据对
data_root = Path("outputs/data/stage2_v2_100k")
pairs = LatentPairs(str(data_root), split="val", sample_count=100, seed=42)

sample_slider = mo.ui.slider(
    start=0,
    stop=len(pairs) - 1,
    value=0,
    label="滑动选择验证集样本序号 (0~99):",
)
mo.vstack([
    mo.md("### 🚗 第二步：从验证集中抽取一对真实的相邻帧 $(t, t+1)$"),
    sample_slider,
])

In [ ]:
sample = pairs[sample_slider.value]
bev_curr = sample["bev"]     # (4, 128, 128)
bev_next = sample["future"]  # (4, 128, 128)
ego_speed = sample["ego"][0] * 10.0   # 真实线速度 (m/s)
yaw_rate = sample["ego"][1]           # 偏航角速度 (rad/s)

img_curr = Image.fromarray(colorize_bev_4ch(bev_curr)).resize((200, 200))
img_next = Image.fromarray(colorize_bev_4ch(bev_next)).resize((200, 200))

mo.vstack([
    mo.hstack([
        mo.vstack([mo.md("**当前帧 BEV ($t$ 时刻)**"), img_curr]),
        mo.vstack([mo.md("**未来真实帧 BEV ($t+1$ 时刻)**"), img_next]),
        mo.vstack([
            mo.md("**自车实时运动状态**"),
            mo.md(f"- **当前车速**: `{ego_speed:.2f} m/s` ({ego_speed * 3.6:.1f} km/h)"),
            mo.md(f"- **角速度 (转向)**: `{yaw_rate:.3f} rad/s`"),
            mo.md(f"- **是否在扰动注入中**: `{sample['noisy']}`"),
            mo.md(f"- **是否在救车接管中**: `{sample['recovery']}`"),
        ]),
    ], gap=3),
    mo.md("💡 *请看上面的两张图：自车在往上开，路面和前车在 $t+1$ 时刻发生微小位移。模型接下来要用 $t$ 时刻和车速，去预测下一刻的隐变量！*"),
])

In [ ]:
# 将 numpy 数组转成 PyTorch Tensor
t_curr = torch.from_numpy(bev_curr).unsqueeze(0)    # (1, 4, 128, 128)
t_next = torch.from_numpy(bev_next).unsqueeze(0)    # (1, 4, 128, 128)
t_ego = torch.from_numpy(sample["ego"]).unsqueeze(0) # (1, 2)

with torch.no_grad():
    # 1. 编码器将当前帧与未来帧分别转为潜空间 Token
    z_curr = model.encoder(t_curr)   # shape: (1, 64, 256)
    z_target = model.encoder(t_next) # shape: (1, 64, 256)

    # 2. 预测器结合自车运动推演未来
    z_pred = model.predictor(z_curr, t_ego) # shape: (1, 64, 256)

In [ ]:
mo.vstack([
    mo.md("### 🧩 第三步：潜空间张量流动过程拆解"),
    mo.md(r"""
    让我们看看这一步里数据的形状（Shape）到底经历了什么变化：
    1. **输入图片**: `[1, 4, 128, 128]` (1张4通道128×128栅格)
    2. **Conv-Stem 卷积下采样**: 
       - 经过两层步长为 2 的卷积，空间压缩到 `[1, 128, 32, 32]`
    3. **Patch 聚合与 Token 化**:
       - 进一步聚合为 **$8 \times 8 = 64$ 个空间 Token**，每个 Token 维度 $d=256$
       - 编码器输出潜变量 $Z_t$: **`[1, 64, 256]`**
    4. **预测器推演**:
       - 输入 $Z_t$ `[1, 64, 256]` + 自车动作 `[1, 2]`
       - 吐出推演出的下一帧 $\hat{Z}_{t+1}$: **`[1, 64, 256]`**
    """),
    mo.md(f"- **真实输出张量尺寸**: 当前帧 `z_curr`: `{list(z_curr.shape)}` | 预测帧 `z_pred`: `{list(z_pred.shape)}`"),
])

In [ ]:
# 模拟计算一个 Batch 的 VICReg 损失
# 单样本无法算批协方差，这里用微扰动构造一个小 batch 演示真实数字
batch_z_curr = torch.cat([z_curr, z_curr + torch.randn_like(z_curr) * 0.1], dim=0)
batch_z_target = torch.cat([z_target, z_target + torch.randn_like(z_target) * 0.1], dim=0)
batch_z_pred = torch.cat([z_pred, z_pred + torch.randn_like(z_pred) * 0.1], dim=0)

loss_total, terms = latent_loss(batch_z_pred, batch_z_target, batch_z_curr, config)

# 计算与“直接复制当前帧”对比的真实 MSE 误差
pred_mse = F.mse_loss(z_pred, z_target).item()
copy_mse = F.mse_loss(z_curr, z_target).item()
gain_pct = (1.0 - pred_mse / max(copy_mse, 1e-12)) * 100

mo.vstack([
    mo.md("### 📐 第四步：手把手算 VICReg 损失（看清每一个数学项）"),
    mo.md(r"""
    VICReg 损失公式为：  
    $$\mathcal{L}_{total} = 25 \times \text{Similarity} + 25 \times \text{Variance} + 1 \times \text{Covariance}$$

    当前样本的真实数值如下：
    - 🎯 **1. 预测相似度 (Similarity MSE)**: """ + f"`{terms['prediction']:.6f}`" + r"""  
      *(越小越好：代表预测隐变量和真实隐变量贴得越近)*
    - 🛡️ **2. 方差惩罚 (Variance)**: """ + f"`{terms['variance']:.6f}`" + r"""  
      *(维持通道标准差 $\ge 1.0$，惩罚值为 0 说明特征极其活跃，绝不塌缩！)*
    - 🔀 **3. 协方差惩罚 (Covariance)**: """ + f"`{terms['covariance']:.6f}`" + r"""  
      *(惩罚非对角线相关性，迫使 256 个通道互相正交独立)*
    - 🏆 **总损失**: """ + f"`{loss_total.item():.4f}`" + r"""
    """),
    mo.md(f"""
    #### 核心基线对比：
    - **预测未来 MSE**: `{pred_mse:.6f}`
    - **复制上一帧 Dummy MSE**: `{copy_mse:.6f}`
    - 🚀 **推演增益**: `{gain_pct:+.1f}%` *(正数代表模型确实在脑补时空演进，而不是简单死记复制上一帧)*
    """),
])

In [ ]:
# 将 256 维抽象特征通过 PCA 投影为 RGB 伪彩色图像，供肉眼直观查看！
def latent_to_rgb(z_tensor):
    # z: (1, 64, 256) -> (64, 256)
    feat = z_tensor.squeeze(0).cpu().numpy()
    # 简单使用前 3 个主要奇异值方向投影到 RGB
    u, s, vt = np.linalg.svd(feat - feat.mean(0), full_matrices=False)
    proj = feat @ vt[:3].T  # (64, 3)
    # 归一化到 0..255
    proj = (proj - proj.min(0)) / np.maximum(proj.max(0) - proj.min(0), 1e-6)
    rgb_tokens = (proj * 255).astype(np.uint8).reshape(8, 8, 3)
    return Image.fromarray(rgb_tokens).resize((160, 160), Image.Resampling.NEAREST)

pca_curr = latent_to_rgb(z_curr)
pca_pred = latent_to_rgb(z_pred)
pca_target = latent_to_rgb(z_target)

mo.vstack([
    mo.md("### 🎨 第五步：把 256 维抽象潜空间画出来给你看（PCA 伪彩色图）"),
    mo.md("因为隐变量是纯数字，我们通过 PCA 把 256 维特征降维到 RGB 3 个通道，每个 $8 \\times 8$ 的马赛克格子代表一个空间区域的特征状态："),
    mo.hstack([
        mo.vstack([mo.md("**当前帧特征状态 ($Z_t$)**"), pca_curr]),
        mo.vstack([mo.md("**模型预测的特征状态 ($\\hat{Z}_{t+1}$)**"), pca_pred]),
        mo.vstack([mo.md("**真实的特征状态 ($Z_{t+1}$)**"), pca_target]),
    ], gap=3),
    mo.md("💡 *你可以清晰看到：中间模型推演出的颜色与分布模式，高度贴合右侧真实的特征图！这证明了时空世界模型在潜空间推演的真实有效性。*"),
])